In [1]:
import os
import re
import csv
import glob
import json
import difflib
from pathlib import Path
from collections import defaultdict


In [ ]:
# =========================
# CONFIG
# =========================
BODY_DIR = r"../annotations/sepsis-10/body"
MODEL_DIRS = {
    "dsr1": r"../annotations/sepsis-40/sepsis-40_dsr1_tts",
    "llama33": r"../annotations/sepsis-40/sepsis-40_l33_tts",
}
OUT_DIR = r"../results/other_experiments/hallucination_audit"

os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
# =========================
# HELPERS
# =========================

def normalize_whitespace(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def normalize_text(s: str) -> str:
    """
    Conservative normalization:
    - lowercase
    - normalize unicode punctuation to spaces
    - keep alphanumerics
    - collapse whitespace
    """
    s = s.lower()
    s = s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    s = s.replace("-", " ")
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def sentence_split(text: str):
    """
    Very simple sentence splitter.
    Good enough for evidence extraction.
    """
    text = text.replace("\r\n", "\n")
    parts = re.split(r'(?<=[\.\?\!])\s+|\n+', text)
    return [normalize_whitespace(p) for p in parts if normalize_whitespace(p)]

def token_list(s: str):
    return [t for t in normalize_text(s).split() if t]

def is_ordered_subsequence(short_tokens, long_tokens):
    """
    Checks whether short_tokens appear in long_tokens in order, not necessarily contiguously.
    """
    if not short_tokens:
        return False
    i = 0
    for tok in long_tokens:
        if tok == short_tokens[i]:
            i += 1
            if i == len(short_tokens):
                return True
    return False

def strip_leading_scaffold(event: str):
    """
    Remove common prompt-driven scaffolding that may not appear literally in text.
    We keep this conservative.
    """
    e = normalize_text(event)

    # common prompt-driven or annotation-style prefixes
    prefixes = [
        "history of ",
        "diagnosis of ",
        "treated with ",
        "treatment with ",
        "received treatment with ",
        "received ",
        "given ",
        "on ",
        "with ",
    ]
    for p in prefixes:
        if e.startswith(p):
            e = e[len(p):].strip()

    return e

def make_event_variants(event: str):
    """
    Generate conservative variants to search.
    These are to reduce false flags caused by prompt instructions,
    not to over-match everything.
    """
    raw = normalize_text(event)
    base = strip_leading_scaffold(event)

    variants = set()
    variants.add(raw)
    variants.add(base)

    # remove trivial auxiliaries that often come from the LLM's phrasing
    # Example: "fever persisted" -> "fever"
    simplified = re.sub(
        r"\b(persisted|persistent|persisting|presented|presenting|developed|diagnosed|diagnosis|treated|treatment|received|receiving|admitted|admission|discharged|discharge)\b",
        " ",
        base
    )
    simplified = normalize_text(simplified)
    if simplified:
        variants.add(simplified)

    # singular/plural-ish simple handling: remove trailing s on longer tokens
    toks = token_list(base)
    if toks:
        toks2 = []
        for t in toks:
            if len(t) > 4 and t.endswith("s"):
                toks2.append(t[:-1])
            else:
                toks2.append(t)
        variants.add(" ".join(toks2))

    # negative finding variants
    # "no shortness of breath", "denies chest pain", "without fever"
    neg_patterns = [
        r"^no\s+(.+)$",
        r"^denies\s+(.+)$",
        r"^denied\s+(.+)$",
        r"^without\s+(.+)$",
    ]
    for pat in neg_patterns:
        m = re.match(pat, base)
        if m:
            core = normalize_text(m.group(1))
            if core:
                variants.add(core)

    # drop stop-like connectors if there are enough content tokens
    stopish = {
        "of", "with", "and", "or", "the", "a", "an", "to", "for", "in", "on", "at",
        "history", "diagnosis", "treatment", "received", "given"
    }
    toks = [t for t in token_list(base) if t not in stopish]
    if len(toks) >= 1:
        variants.add(" ".join(toks))

    variants = {v for v in variants if v and len(v) >= 2}
    return sorted(variants, key=lambda x: (-len(x), x))

def find_best_match(event: str, body_text: str):
    """
    Returns a dict:
    {
      found: bool,
      match_type: str,
      matched_variant: str,
      evidence: str,
      sentence_idx: int or None,
      similarity: float or None
    }

    Matching order:
    1) exact normalized substring in full text
    2) exact normalized substring in sentence
    3) token subsequence in sentence
    4) all tokens present in sentence (unordered) for informative review support
    5) fuzzy nearest sentence for unresolved manual review
    """
    norm_body = normalize_text(body_text)
    sentences = sentence_split(body_text)
    norm_sentences = [normalize_text(s) for s in sentences]

    variants = make_event_variants(event)

    # 1) exact substring in full document
    for v in variants:
        if v in norm_body:
            # find supporting sentence
            for i, ns in enumerate(norm_sentences):
                if v in ns:
                    return {
                        "found": True,
                        "match_type": "exact_substring_sentence",
                        "matched_variant": v,
                        "evidence": sentences[i],
                        "sentence_idx": i,
                        "similarity": 1.0,
                    }
            return {
                "found": True,
                "match_type": "exact_substring_document",
                "matched_variant": v,
                "evidence": "",
                "sentence_idx": None,
                "similarity": 1.0,
            }

    # 2) ordered token subsequence in a sentence
    for v in variants:
        v_toks = token_list(v)
        if not v_toks:
            continue
        for i, ns in enumerate(norm_sentences):
            s_toks = ns.split()
            if is_ordered_subsequence(v_toks, s_toks):
                return {
                    "found": True,
                    "match_type": "ordered_token_subsequence",
                    "matched_variant": v,
                    "evidence": sentences[i],
                    "sentence_idx": i,
                    "similarity": None,
                }

    # 3) unordered token containment in a sentence
    # useful for cases where wording is broken up
    for v in variants:
        v_toks = set(token_list(v))
        if not v_toks:
            continue
        for i, ns in enumerate(norm_sentences):
            s_toks = set(ns.split())
            if v_toks.issubset(s_toks):
                return {
                    "found": True,
                    "match_type": "unordered_token_containment",
                    "matched_variant": v,
                    "evidence": sentences[i],
                    "sentence_idx": i,
                    "similarity": None,
                }

    # 4) unresolved: attach nearest sentence by fuzzy score for manual review
    best_score = -1.0
    best_sent = ""
    best_idx = None
    best_var = ""

    for v in variants:
        for i, ns in enumerate(norm_sentences):
            score = difflib.SequenceMatcher(None, v, ns).ratio()
            if score > best_score:
                best_score = score
                best_sent = sentences[i]
                best_idx = i
                best_var = v

    return {
        "found": False,
        "match_type": "not_found_auto",
        "matched_variant": best_var,
        "evidence": best_sent,
        "sentence_idx": best_idx,
        "similarity": round(best_score, 4) if best_score >= 0 else None,
    }

def read_annotation_file(path):
    """
    Reads either:
    - pipe-separated files like: event | time
    - csv files with columns event,time
    Returns list of dicts with keys: event, time
    """
    rows = []

    # first try csv sniff
    with open(path, "r", encoding="utf-8-sig", errors="ignore") as f:
        sample = f.read(4096)
        f.seek(0)

        # heuristic: many of your files may actually be pipe-separated despite .csv
        if "|" in sample and sample.count("|") >= sample.count(","):
            for line in f:
                line = line.strip()
                if not line:
                    continue
                if "|" not in line:
                    continue
                parts = [p.strip() for p in line.split("|", 1)]
                if len(parts) != 2:
                    continue
                event, time_val = parts
                if event.lower() == "event" and time_val.lower() == "time":
                    continue
                rows.append({"event": event, "time": time_val})
            return rows

        # standard csv
        f.seek(0)
        reader = csv.DictReader(f)
        cols = [c.lower().strip() for c in (reader.fieldnames or [])]
        if "event" in cols and "time" in cols:
            # map back to original header names
            event_col = reader.fieldnames[cols.index("event")]
            time_col = reader.fieldnames[cols.index("time")]
            for r in reader:
                event = (r.get(event_col) or "").strip()
                time_val = (r.get(time_col) or "").strip()
                if event:
                    rows.append({"event": event, "time": time_val})
            return rows

    # final fallback: raw line parsing
    with open(path, "r", encoding="utf-8-sig", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if "|" in line:
                parts = [p.strip() for p in line.split("|", 1)]
                if len(parts) == 2:
                    rows.append({"event": parts[0], "time": parts[1]})
    return rows

def build_body_index(body_dir):
    """
    Map file stem -> txt path
    """
    out = {}
    for p in glob.glob(os.path.join(body_dir, "*.txt")):
        stem = Path(p).stem
        out[stem] = p
    return out

def build_annotation_index(model_dir):
    """
    Map file stem -> annotation path
    """
    out = {}
    for p in glob.glob(os.path.join(model_dir, "*")):
        if os.path.isdir(p):
            continue
        stem = Path(p).stem
        # keep only plausible csv/txt/bsv-ish files
        if Path(p).suffix.lower() in {".csv", ".txt", ".tsv", ".bsV".lower()} or True:
            out[stem] = p
    return out

In [4]:
# =========================
# MAIN AUDIT
# =========================

body_index = build_body_index(BODY_DIR)

all_rows = []
summary_rows = []

for model_name, model_dir in MODEL_DIRS.items():
    ann_index = build_annotation_index(model_dir)

    overlap = sorted(set(body_index.keys()) & set(ann_index.keys()))
    if not overlap:
        print(f"[WARN] No overlapping files found for model={model_name}")
        continue

    print(f"[INFO] model={model_name} overlap_n={len(overlap)}")

    model_total = 0
    model_auto_found = 0
    model_needs_review = 0

    for stem in overlap:
        body_path = body_index[stem]
        ann_path = ann_index[stem]

        with open(body_path, "r", encoding="utf-8-sig", errors="ignore") as f:
            body_text = f.read()

        ann_rows = read_annotation_file(ann_path)

        case_total = 0
        case_auto_found = 0
        case_needs_review = 0

        for idx, row in enumerate(ann_rows):
            event = row["event"].strip()
            time_val = row["time"]

            if not event:
                continue

            res = find_best_match(event, body_text)

            out_row = {
                "model": model_name,
                "case_id": stem,
                "annotation_file": ann_path,
                "body_file": body_path,
                "row_idx": idx,
                "event": event,
                "time": time_val,
                "auto_found": int(res["found"]),
                "match_type": res["match_type"],
                "matched_variant": res["matched_variant"],
                "evidence_sentence_idx": res["sentence_idx"],
                "evidence_text": res["evidence"],
                "similarity_if_unfound": res["similarity"],
                # fill manually later if you want
                "manual_label": "",  # supported / hallucinated / unsure
                "manual_notes": "",
            }
            all_rows.append(out_row)

            case_total += 1
            model_total += 1
            if res["found"]:
                case_auto_found += 1
                model_auto_found += 1
            else:
                case_needs_review += 1
                model_needs_review += 1

        summary_rows.append({
            "level": "case",
            "model": model_name,
            "case_id": stem,
            "n_events": case_total,
            "n_auto_found": case_auto_found,
            "n_needs_manual_review": case_needs_review,
            "auto_found_rate": round(case_auto_found / case_total, 4) if case_total else None,
        })

    summary_rows.append({
        "level": "model",
        "model": model_name,
        "case_id": "",
        "n_events": model_total,
        "n_auto_found": model_auto_found,
        "n_needs_manual_review": model_needs_review,
        "auto_found_rate": round(model_auto_found / model_total, 4) if model_total else None,
    })

[INFO] model=dsr1 overlap_n=10
[INFO] model=llama33 overlap_n=10


In [ ]:
# =========================
# SAVE OUTPUTS
# =========================

audit_csv = os.path.join(OUT_DIR, "hallucination_audit_all_rows.csv")
review_csv = os.path.join(OUT_DIR, "hallucination_audit_manual_review_only.csv")
summary_csv = os.path.join(OUT_DIR, "hallucination_audit_summary.csv")
meta_json = os.path.join(OUT_DIR, "hallucination_audit_metadata.json")

fieldnames = [
    "model", "case_id", "annotation_file", "body_file", "row_idx",
    "event", "time",
    "auto_found", "match_type", "matched_variant",
    "evidence_sentence_idx", "evidence_text",
    "similarity_if_unfound",
    "manual_label", "manual_notes"
]

with open(audit_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in all_rows:
        writer.writerow(r)

with open(review_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in all_rows:
        if int(r["auto_found"]) == 0:
            writer.writerow(r)

summary_fields = [
    "level", "model", "case_id", "n_events", "n_auto_found",
    "n_needs_manual_review", "auto_found_rate"
]
with open(summary_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=summary_fields)
    writer.writeheader()
    for r in summary_rows:
        writer.writerow(r)

metadata = {
    "body_dir": BODY_DIR,
    "model_dirs": MODEL_DIRS,
    "n_total_rows": len(all_rows),
    "n_manual_review_rows": sum(1 for r in all_rows if int(r["auto_found"]) == 0),
    "notes": [
        "This audit is conservative.",
        "auto_found=1 means the event text was grounded automatically by exact or near-exact lexical evidence.",
        "auto_found=0 does NOT imply hallucination; it means manual review is required.",
        "manual_label can be filled with: supported / hallucinated / unsure"
    ]
}
with open(meta_json, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"[DONE] Wrote:\n- {audit_csv}\n- {review_csv}\n- {summary_csv}\n- {meta_json}")